> **Chapter 7, Part 5** | Bridge notebook. **Focus:** chunking, metadata survival, and the move from vector search to usable retrieval.


# Chunking, Metadata, and Retrieval Bridges

This notebook connects the vector index to the retrieval system.

Embeddings alone do not answer a question. The system still needs chunk boundaries, parent identifiers, metadata that survive indexing, and a result shape that can later produce citations.

## Outputs

- a small chunking routine
- chunk records with parent IDs and metadata
- a toy search that returns citation-ready payloads
- a bridge into Chapter 10.1 and 10.4

## Supporting reading

- Chapter 10.1 for system contracts
- Chapter 10.4 for the local vector store implementation
- LangChain text splitter reference for comparison: https://python.langchain.com/docs/concepts/text_splitters/

## Failure note

If chunk metadata disappears at indexing time, you usually discover it too late, when filters or citations stop working.

## How I would debug this

I inspect one content item, one chunk list, one stored row, and one ranked result side by side. If any field vanishes in that chain, the later answer layer will have weak footing.


In [ ]:
import math
import pandas as pd

contents = [
    {
        "source_id": "parks-note",
        "title": "Accessible trail notes",
        "body": "The riverside stop has a paved route, shuttle access, and clear signage. Families use it as a first stop.",
        "metadata": {"domain": "parks", "state": "UT"},
    },
    {
        "source_id": "gov-note",
        "title": "Golden record policy",
        "body": "A golden record should preserve source lineage, survivorship logic, and steward review when entity boundaries remain unstable.",
        "metadata": {"domain": "governance", "state": "NA"},
    },
]


def chunk_text(item, chunk_size=10):
    words = item["body"].split()
    rows = []
    for idx in range(0, len(words), chunk_size):
        text = " ".join(words[idx: idx + chunk_size])
        rows.append(
            {
                "chunk_id": f"{item['source_id']}-chunk-{idx // chunk_size + 1}",
                "parent_id": item["source_id"],
                "title": item["title"],
                "text": text,
                "metadata": item["metadata"],
            }
        )
    return rows

chunks = [chunk for item in contents for chunk in chunk_text(item)]
pd.DataFrame(chunks)


In [ ]:
vocab = {
    "accessible": [1.0, 0.1, 0.0],
    "trail": [0.9, 0.2, 0.0],
    "shuttle": [0.8, 0.3, 0.0],
    "golden": [0.0, 0.2, 1.0],
    "record": [0.0, 0.1, 0.9],
    "steward": [0.1, 0.0, 0.8],
    "unstable": [0.0, 0.0, 1.0],
}


def embed_text(text):
    vector = [0.0, 0.0, 0.0]
    for token in text.lower().replace(',', '').replace('.', '').split():
        if token in vocab:
            vector = [left + right for left, right in zip(vector, vocab[token])]
    norm = math.sqrt(sum(value * value for value in vector)) or 1.0
    return [value / norm for value in vector]


for chunk in chunks:
    chunk["vector"] = embed_text(chunk["text"])

pd.DataFrame(chunks)[["chunk_id", "parent_id", "text", "metadata"]]


In [ ]:
def cosine(left, right):
    numerator = sum(a * b for a, b in zip(left, right))
    left_norm = math.sqrt(sum(a * a for a in left)) or 1.0
    right_norm = math.sqrt(sum(b * b for b in right)) or 1.0
    return numerator / (left_norm * right_norm)


def search_chunks(query, rows, filters=None, top_k=3):
    query_vector = embed_text(query)
    matches = []
    for row in rows:
        if filters:
            for key, value in filters.items():
                if row["metadata"].get(key) != value:
                    break
            else:
                matches.append(row)
        else:
            matches.append(row)

    ranked = sorted(
        [
            {
                "chunk_id": row["chunk_id"],
                "parent_id": row["parent_id"],
                "title": row["title"],
                "snippet": row["text"],
                "domain": row["metadata"]["domain"],
                "score": cosine(query_vector, row["vector"]),
            }
            for row in matches
        ],
        key=lambda row: row["score"],
        reverse=True,
    )
    return pd.DataFrame(ranked[:top_k])


search_chunks("accessible shuttle first stop", chunks)


In [ ]:
search_chunks("unstable golden record steward review", chunks, filters={"domain": "governance"})


## Why this is the actual bridge

The result already looks more like Chapter 10 than like an isolated embedding demo. We now have:

- parent IDs
- chunk IDs
- snippets
- metadata filters
- retrieval scores

That is the transition learners need before they meet a full retrieval system with a real vector backend and answer layer.

## Exercise

1. add a `source_url` field and return it in the search result
2. increase the chunk size and observe how the ranked snippet quality changes
3. create a third content item from a different domain and show why metadata filtering becomes more valuable as the corpus grows
